# Import Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Define chromosome start and end positions (found from external data sources). Randomly generate origin positions between these boundaries.

This code generates random replication origin positions across yeast chromosomes based on their lengths. It first reads chromosome sizes and origin data, then randomly places the same number of origins per chromosome as in the actual dataset (optionally filtering only confirmed origins). Finally, it plots a histogram showing the distribution of these random origin positions across all chromosomes.


In [ ]:
# Read the yeast replication origin data and chromosome lengths.
cer_data = pd.read_csv('cerevisiae-data.csv')
# Assume chromosome_length.csv has no header; columns: [chrom, accession, length]
chr_size = pd.read_csv('chromosome_length.csv', header=None, names=['chr', 'accession', 'length'])

# Convert chromosome column to numeric (if needed)
cer_data['chr'] = pd.to_numeric(cer_data['chr'], errors='coerce')
chr_size['chr'] = pd.to_numeric(chr_size['chr'], errors='coerce')

def generate_random_origins(cer_data, chr_size, confirmed_only=False):
    """
    For each chromosome, use the chromosome length (from chr_size) and generate a set of random origin positions.
    The number of random points equals the number of replication origin points in cer_data (or confirmed ones if specified).
    """
    random_origins_list = []
    chromosomes = sorted(cer_data['chr'].unique())

    for chrom in chromosomes:
        # Get chromosome length from chr_size:
        chrom_length = int(chr_size[chr_size['chr'] == chrom]['length'].values[0])
        start_boundary = 1
        end_boundary = chrom_length

        # If confirmed_only is True, count only origins with status 'Confirmed'
        if confirmed_only:
            num_points = len(cer_data[(cer_data['chr'] == chrom) & (cer_data['status'] == 'Confirmed')])
        else:
            num_points = len(cer_data[cer_data['chr'] == chrom])

        # Generate random origins; if no origins present, skip.
        if num_points == 0:
            continue
        rand_positions = np.sort(np.random.randint(start_boundary, end_boundary+1, num_points))
        # Save each random point with its chromosome label
        for pos in rand_positions:
            random_origins_list.append({'chr': chrom, 'random_origin': pos})

    return pd.DataFrame(random_origins_list)

# Generate random origins for all entries (or set confirmed_only=True to use only confirmed origins)
rand_origins_df = generate_random_origins(cer_data, chr_size, confirmed_only=False)
print("Random origins (first 10 rows):")
print(rand_origins_df.head(10))

# Plot histogram of all generated random origins across chromosomes.
plt.figure(figsize=(10, 6))
sns.histplot(rand_origins_df['random_origin'], bins=50, color='skyblue', label='Random Origins')
plt.xlabel("Position")
plt.ylabel("Frequency")
plt.title("Histogram of Randomly Generated Origins (Task 11)")
plt.legend()
plt.show()

This code filters confirmed replication origin regions from genomic data, calculates their midpoints and average sizes, and simulates random origin points within each region. It then plots and compares the observed midpoints and simulated positions to analyze the distribution and randomness of origin placement.

In [ ]:
# Filter confirmed regions
confirmed = cer_data[cer_data['status'] == 'Confirmed'].copy()

# Calculate midpoints and region sizes
confirmed['mid_point'] = (confirmed['start'] + confirmed['end']) / 2
confirmed['region_size'] = confirmed['end'] - confirmed['start']
avg_size = confirmed['region_size'].mean()

print(f"Average Size of Confirmed Origins: {avg_size}")
print(f"Genome span: {confirmed['end'].max() - confirmed['start'].min()}")
print(f"Number of confirmed origins: {len(confirmed)}")
print(f"Required space for all origins: {avg_size * (len(confirmed) - 1)}")

# Function: Simulate one point within each confirmed region
def simulate_within_regions(confirmed):
    return np.array([
        np.random.randint(row['start'], row['end'] + 1)
        for _, row in confirmed.iterrows()
    ])

# Simulate once (we don’t need spacing logic anymore)
simulated_points = simulate_within_regions(confirmed)

# Plot: Observed midpoints
plt.figure(figsize=(12, 6))
sns.histplot(confirmed['mid_point'], bins=30, stat='density', kde=True, color='blue', label='Observed (Midpoints)')
plt.xlabel('Position')
plt.ylabel('Density')
plt.title('Observed Confirmed Origin Midpoints')
plt.legend()
plt.tight_layout()
plt.show()

# Plot: Simulated points
plt.figure(figsize=(12, 6))
sns.histplot(simulated_points, bins=30, stat='density', kde=True, color='orange', label='Simulated Origins')
plt.xlabel('Position')
plt.ylabel('Density')
plt.title('Simulated Origin Positions (One per Confirmed Region)')
plt.legend()
plt.tight_layout()
plt.show()